# Process Library Runner: Process-Agnostic Batch-First Aspen Workflow

Use this notebook to gather evidence for a user-defined chemical process, prepare Codex-ready YAML generation artifacts, validate `process.yaml`, and run the generic Aspen batch-first Gate 1/Gate 2 workflow. Example-specific diagnostics and tuning live outside this generic workflow.


## 1. Environment setup and imports


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "aspen_automation").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook working directory.")

REPO_ROOT = find_repo_root(Path.cwd()).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from aspen_automation import (
    analyze_process_spec_coherence,
    apply_process_spec_improvements,
    build_codex_improvement_markdown,
    build_codex_results_markdown,
    build_codex_spec_markdown,
    build_process_intake_artifacts,
    check_aspen_running,
    generate_inp,
    load_process_spec,
    load_result_artifact_tables,
    load_spec,
    run_aspen_batch,
    run_process_batch_first,
    scan_process_library,
    suggest_process_spec_improvements,
    validate_process_spec_file,
    write_process_spec_file,
)

print(f"Repository root: {REPO_ROOT}")


## 1.5 Aspen Plus pre-flight check

Launch Aspen Plus from AppsAnywhere/Porticada before running live Gate 1 or Gate 2 cells.


In [ ]:
try:
    aspen_preflight = check_aspen_running()
except Exception as exc:
    aspen_preflight = {"aspen_running": False, "error": str(exc)}
display(pd.DataFrame([aspen_preflight]))


## 2. Process evidence intake for Codex-authored YAML

Fill these fields for a new process. The notebook writes evidence artifacts only; it does not perform web searches, read PDFs, or call an LLM. Pass the generated prompt file to Codex to create or revise `process.yaml`.


In [ ]:
PROCESS_NAME = ""  # Example: "ammonia" or "hydrogen_reforming"
USER_PROCESS_BRIEF = """
Describe the desired chemical process, feeds, products, capacity, operating envelope, and modeling objective.
"""
SOURCE_PDFS: list[str] = []
SOURCE_URLS: list[str] = []
WEB_SEARCH_QUERIES: list[str] = []
REFERENCE_NOTES: list[str] = []
WRITE_PROCESS_INTAKE_ARTIFACTS = False

intake_artifacts = None
if WRITE_PROCESS_INTAKE_ARTIFACTS:
    if not PROCESS_NAME.strip():
        raise ValueError("Set PROCESS_NAME before writing intake artifacts.")
    intake_artifacts = build_process_intake_artifacts(
        PROCESS_NAME,
        REPO_ROOT / "process_library",
        user_process_brief=USER_PROCESS_BRIEF,
        source_pdfs=SOURCE_PDFS,
        source_urls=SOURCE_URLS,
        web_search_queries=WEB_SEARCH_QUERIES,
        reference_notes=REFERENCE_NOTES,
    )
    display(pd.DataFrame([{
        "process_name": intake_artifacts.process_name,
        "source_manifest.json": str(intake_artifacts.source_manifest_path),
        "process_research_brief.md": str(intake_artifacts.research_brief_path),
        "codex_process_yaml_prompt.md": str(intake_artifacts.codex_prompt_path),
    }]))
else:
    print("Set WRITE_PROCESS_INTAKE_ARTIFACTS=True to create source_manifest.json, process_research_brief.md, and codex_process_yaml_prompt.md.")


## 3. Process library path configuration


In [ ]:
PROCESS_LIBRARY_DIR = REPO_ROOT / "process_library"
PROCESS_RUNS_DIR = REPO_ROOT / "process_runs" / "batch_first_capsule"
GATE1_BATCH_ROOT = PROCESS_RUNS_DIR / "gate1_batch_translator"

LIBRARY_ROOT = PROCESS_LIBRARY_DIR
RUNS_ROOT = PROCESS_RUNS_DIR
VISIBLE = True
TIMEOUT_SECONDS = 1800
BATCH_TIMEOUT_SECONDS = 1800
REPORT_FORMAT = "html"
ENFORCE_ACCEPTANCE_TARGETS = False
RUN_GATE1 = True
RUN_GATE2 = True
AUTO_APPLY_SUGGESTED_IMPROVEMENTS = False
ONLY_PROCESSES: set[str] | None = None

print(f"Process library: {LIBRARY_ROOT}")
print(f"Batch-first process runs: {RUNS_ROOT}")
print(f"Gate 1 batch translator workspace: {GATE1_BATCH_ROOT}")
print(f"ENFORCE_ACCEPTANCE_TARGETS={ENFORCE_ACCEPTANCE_TARGETS}")
print(f"ONLY_PROCESSES={ONLY_PROCESSES}")


## 4. YAML schema / expected fields overview

Each process folder must contain `process.yaml` using the supported process-library schema: `metadata`, `components`, `properties`, `flowsheet`, `streams`, and `blocks`, with optional `chemistry`, `reaction_sets`, `kinetic_models`, `process_defaults`, and `targets`.


## 5. Discovery of all available process folders


In [ ]:
scan = scan_process_library(LIBRARY_ROOT)
candidate_processes = [
    process for process in scan.processes
    if ONLY_PROCESSES is None or process.name in ONLY_PROCESSES
]

display(pd.DataFrame([
    {"process_name": process.name, "process_dir": str(process.process_dir), "spec_path": str(process.spec_path)}
    for process in candidate_processes
]))
if scan.issues:
    display(pd.DataFrame([issue.__dict__ for issue in scan.issues]))
print(f"Discovered {len(candidate_processes)} runnable process folder(s).")


## 6. Shared helper functions


In [ ]:
def read_json_file(path: Path) -> dict:
    if not path.exists():
        return {"missing": str(path)}
    return json.loads(path.read_text(encoding="utf-8"))

def history_blocking_messages(batch_result, limit: int = 5) -> list[str]:
    diagnostics = getattr(batch_result, "history_diagnostics", None) or {}
    messages = diagnostics.get("blocking_messages") or diagnostics.get("messages") or []
    rendered: list[str] = []
    for message in messages[:limit]:
        if isinstance(message, dict):
            rendered.append(str(message.get("message") or message))
        else:
            rendered.append(str(message))
    return rendered

def batch_history_path(batch_result) -> str | None:
    explicit_path = getattr(batch_result, "history_path", None)
    if explicit_path:
        return str(explicit_path)
    artifact = (getattr(batch_result, "artifacts", None) or {}).get(".his", {})
    path = artifact.get("path")
    return str(path) if path else None


## 7. YAML coherence analysis per discovered process


In [ ]:
validation_reports: dict[str, dict] = {}
coherence_reports: dict[str, dict] = {}
coherent_processes = []

for process in candidate_processes:
    print(f"Process directory: {process.process_dir}")
    validation_report = validate_process_spec_file(process.spec_path)
    validation_reports[process.name] = validation_report
    if not validation_report.get("valid"):
        print(f"Skipping coherence because validation failed for {process.name}.")
        display(pd.DataFrame(validation_report.get("errors", [])))
        continue
    spec = load_process_spec(process.process_dir)
    coherence_report = analyze_process_spec_coherence(spec)
    coherence_reports[process.name] = coherence_report
    display(Markdown(build_codex_spec_markdown(process.name, process.spec_path, validation_report, coherence_report)))
    if coherence_report.get("passed"):
        coherent_processes.append(process)

display(pd.DataFrame([
    {
        "process_name": process.name,
        "validation_valid": validation_reports.get(process.name, {}).get("valid"),
        "coherence_passed": coherence_reports.get(process.name, {}).get("passed"),
    }
    for process in candidate_processes
]))


## 8. Suggested YAML improvements per discovered process

Suggestions are displayed by default. Set `AUTO_APPLY_SUGGESTED_IMPROVEMENTS=True` only when you intentionally want the notebook to update `process.yaml`.


In [ ]:
for process in candidate_processes:
    report = coherence_reports.get(process.name)
    if not report:
        continue
    spec = load_process_spec(process.process_dir)
    suggestions = suggest_process_spec_improvements(spec, report)
    if suggestions:
        display(Markdown(build_codex_improvement_markdown(process.name, suggestions)))
    if suggestions and AUTO_APPLY_SUGGESTED_IMPROVEMENTS:
        improved = apply_process_spec_improvements(spec, suggestions)
        write_process_spec_file(process.spec_path, improved, backup=True)
        print(f"Updated {process.spec_path}")


## 9. Gate 1: Aspen batch translator

This gate compiles generated INP with Aspen batch and treats the `.his` file as the source of truth. A `.bkp` alone is not enough.


In [ ]:
gate1_results: dict[str, object] = {}
gate1_ready_processes = []

if not RUN_GATE1:
    gate1_ready_processes = list(coherent_processes)
    print("RUN_GATE1=False; carrying coherent processes forward without batch translation.")
else:
    for process in coherent_processes:
        print(f"Gate 1 process directory: {process.process_dir}")
        gate1_dir = GATE1_BATCH_ROOT / process.name
        gate1_dir.mkdir(parents=True, exist_ok=True)
        inp_path = gate1_dir / f"{process.name}_generated.inp"
        spec = load_spec(process.spec_path)
        generate_inp(spec, output_path=inp_path)
        batch_result = run_aspen_batch(
            inp_path,
            gate1_dir / "batch",
            run_id=process.name,
            timeout_seconds=BATCH_TIMEOUT_SECONDS,
        )
        gate1_results[process.name] = batch_result
        print(f"Gate 1 succeeded: {batch_result.succeeded}")
        print(f"History status: {(batch_result.history_diagnostics or {}).get('status')}")
        print(f"Archive path: {batch_result.archive_path}")
        history_path = batch_history_path(batch_result)
        print(f"History path: {history_path}")
        print(f"Stdout path: {batch_result.stdout_path}")
        print(f"Stderr path: {batch_result.stderr_path}")
        blocking = history_blocking_messages(batch_result)
        if blocking:
            print("First blocking history messages:")
            for message in blocking:
                print(f"- {message}")
        if batch_result.succeeded:
            gate1_ready_processes.append(process)

display(pd.DataFrame([
    {
        "process_name": name,
        "succeeded": result.succeeded,
        "history_status": (result.history_diagnostics or {}).get("status"),
        "archive_path": str(result.archive_path) if result.archive_path else None,
        "history_path": batch_history_path(result),
    }
    for name, result in gate1_results.items()
]))


## 10. Gate 2: BKP COM load, extraction, and reports

This gate runs the generic batch-first capsule path: generated INP, Aspen batch `.bkp`, COM `InitFromArchive2`, convergence run, result extraction, and report generation. Acceptance targets are displayed but not enforced by default.


In [ ]:
process_results: dict[str, object] = {}

if not RUN_GATE2:
    print("RUN_GATE2=False; skipping BKP COM load/extraction.")
else:
    for process in gate1_ready_processes:
        print(f"Process directory: {process.process_dir}")
        result = run_process_batch_first(
            process.process_dir,
            RUNS_ROOT,
            visible=VISIBLE,
            enforce_acceptance_targets=ENFORCE_ACCEPTANCE_TARGETS,
            timeout_seconds=TIMEOUT_SECONDS,
            batch_timeout_seconds=BATCH_TIMEOUT_SECONDS,
            report_format=REPORT_FORMAT,
        )
        process_results[process.name] = result
        print(f"Status: {result.status}")
        print(f"Succeeded: {result.succeeded}")
        if result.error:
            print(f"Error: {result.error}")
        if result.layout:
            print(f"Run directory: {result.layout.run_dir}")
            print(f"Results directory: {result.layout.results_dir}")

display(pd.DataFrame([
    {
        "process_name": name,
        "status": result.status,
        "succeeded": result.succeeded,
        "error": result.error,
    }
    for name, result in process_results.items()
]))


## 11. Gate 2 diagnostics and evidence bundle

Display `context_probe.json`, `build_diagnostics.json`, `simulation_diagnostics.json`, `acceptance.json`, and `live_aspen_summary.json` for completed runs.


In [ ]:
diagnostic_rows: list[dict[str, object]] = []
for name, result in process_results.items():
    if not getattr(result, "layout", None):
        continue
    results_dir = result.layout.results_dir
    run_dir = result.layout.run_dir
    context_probe = read_json_file(results_dir / "context_probe.json")
    build_diagnostics = read_json_file(results_dir / "build_diagnostics.json")
    simulation_diagnostics = read_json_file(results_dir / "simulation_diagnostics.json")
    acceptance = read_json_file(results_dir / "acceptance.json")
    live_summary = {
        "process_name": name,
        "run_dir": str(run_dir),
        "results_dir": str(results_dir),
        "context_probe_path": str(results_dir / "context_probe.json"),
        "build_diagnostics_path": str(results_dir / "build_diagnostics.json"),
        "simulation_diagnostics_path": str(results_dir / "simulation_diagnostics.json"),
        "acceptance_path": str(results_dir / "acceptance.json"),
        "generated_inp_path": str(result.layout.generated_inp_path),
        "report_dir": str(result.report_dir) if result.report_dir else None,
        "model_quality_warnings": simulation_diagnostics.get("model_quality_warnings", []),
        "nrtl_binary_parameters_status": simulation_diagnostics.get("nrtl_binary_parameters_status"),
        "result_csvs": sorted(str(path) for path in results_dir.glob("*.csv")),
    }
    (run_dir / "live_aspen_summary.json").write_text(json.dumps(live_summary, indent=2), encoding="utf-8")
    diagnostic_rows.append({
        "process_name": name,
        "context_probe.json": str(results_dir / "context_probe.json"),
        "build_diagnostics.json": str(results_dir / "build_diagnostics.json"),
        "simulation_diagnostics.json": str(results_dir / "simulation_diagnostics.json"),
        "acceptance.json": str(results_dir / "acceptance.json"),
        "live_aspen_summary.json": str(run_dir / "live_aspen_summary.json"),
        "context_probe_status": context_probe.get("status") or context_probe.get("missing"),
        "build_status": build_diagnostics.get("status"),
        "simulation_status": simulation_diagnostics.get("convergence_status"),
        "nrtl_binary_parameters_status": simulation_diagnostics.get("nrtl_binary_parameters_status"),
        "model_quality_warning_count": len(simulation_diagnostics.get("model_quality_warnings", [])),
        "acceptance_passed": acceptance.get("passed"),
    })
display(pd.DataFrame(diagnostic_rows))


## 12. Codex session analysis per discovered process

This readout uses CSV artifacts as the source of truth and stays process-agnostic.


In [ ]:
for name, result in process_results.items():
    if not result.succeeded:
        print(f"Skipping analysis for {name} because status={result.status!r}.")
        continue
    artifact_paths, tables = load_result_artifact_tables(result)
    acceptance = read_json_file(result.layout.results_dir / "acceptance.json") if result.layout else {}
    display(Markdown(build_codex_results_markdown(name, artifact_paths, tables, acceptance=acceptance)))


## 13. Output summary and validation


In [ ]:
summary_rows = [
    {
        "process_name": name,
        "status": result.status,
        "succeeded": result.succeeded,
        "run_dir": str(result.layout.run_dir) if result.layout else None,
        "generated_files": ", ".join(result.generated_files),
    }
    for name, result in process_results.items()
]
display(pd.DataFrame(summary_rows))
